**Important** Due to the following warning: *UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10*

Just for this dataset the number of fold has been reduced to 5

In [8]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd

data = pd.read_csv("Data/Yeast.csv")
data

,mcg,gvh,alm,mit,erl,pox,vac,nuc,name
0,0.58,0.61,0.47,0.13,0.5,0.0,0.48,0.22,MIT
1,0.43,0.67,0.48,0.27,0.5,0.0,0.53,0.22,MIT
2,0.64,0.62,0.49,0.15,0.5,0.0,0.53,0.22,MIT
3,0.58,0.44,0.57,0.13,0.5,0.0,0.54,0.22,NUC
4,0.42,0.44,0.48,0.54,0.5,0.0,0.48,0.22,MIT
...,...,...,...,...,...,...,...,...,...
1479,0.81,0.62,0.43,0.17,0.5,0.0,0.53,0.22,ME2
1480,0.47,0.43,0.61,0.40,0.5,0.0,0.48,0.47,NUC
1481,0.67,0.57,0.36,0.19,0.5,0.0,0.56,0.22,ME2
1482,0.43,0.40,0.60,0.16,0.5,0.0,0.53,0.39,NUC


In [9]:
y = data["name"]
X = data.drop(columns="name").copy()

In [10]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

-----

# LDA+GBM and PCA+GBM

In [4]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Any, Dict

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# ---------------------------------------------------------------------
# DuoBoostCV — evaluates only LDA+GB and PCA+GB
# ---------------------------------------------------------------------
@dataclass
class DuoBoostCV:
    """Compare LDA+GradientBoost vs PCA+GradientBoost using cross-validation.

    Parameters
    ----------
    seed : int, default=42
        Seed used both for CV (if not specified) and for the models.
    """

    seed: int = 42
    _models: Dict[str, Any] = field(init=False, repr=False)

    # -----------------------------------------------------------
    # Build models based on the number of features
    # -----------------------------------------------------------
    def _build_models(self, n_features: int) -> Dict[str, Any]:
        half_components = max(1, n_features // 2)
        return {
            "PCA + Gradient Boost": Pipeline(
                [
                    ("pca", PCA(n_components=half_components, random_state=self.seed)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
            "LDA + Gradient Boost": Pipeline(
                [
                    ("lda", LDA(n_components=None)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
        }

    # -----------------------------------------------------------
    # Public API — accepts X, y + all cross_validate parameters
    # -----------------------------------------------------------
    def fit(self, X, y, **cv_kwargs) -> pd.DataFrame:
        X = np.asarray(X)
        y = np.asarray(y)

        cv_kwargs = cv_kwargs.copy()
        cv_kwargs.setdefault("scoring", "accuracy")

        self._models = self._build_models(X.shape[1])
        scores: Dict[str, Dict[str, float]] = {}

        for name, model in self._models.items():
            cv_res = cross_validate(model, X, y, **cv_kwargs)

            # <<< dynamic fix >>>
            metric_key = next(k for k in cv_res if k.startswith("test_"))
            metric_name = metric_key.replace("test_", "")

            scores[name] = {
                f"mean_{metric_name}": cv_res[metric_key].mean(),
                f"std_{metric_name}":  cv_res[metric_key].std(),
            }

        self.results_ = pd.DataFrame(scores).T.sort_values(
            by=f"mean_{metric_name}", ascending=False
        )
        return self.results_
    
    # convenience
    def __call__(self, X, y, **cv_kwargs) -> pd.DataFrame:
        return self.fit(X, y, **cv_kwargs)

    def __repr__(self) -> str:  # pragma: no cover
        rep = f"{self.__class__.__name__}(seed={self.seed})"
        if hasattr(self, "results_"):
            rep += f"\nResults:\n{self.results_.round(4)}"
        return rep
    
trainer = DuoBoostCV(seed=42)

# same arguments you’d pass to cross_validate
results = trainer(
    X, y_encoded,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

print(results)

                      mean_score  std_score
LDA + Gradient Boost    0.569419   0.035125
PCA + Gradient Boost    0.504056   0.023185


-----

# GBM

In [5]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# Define the model
gb_clf = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42
)

# Perform 5-fold cross-validation and compute accuracy
cv_scores = cross_val_score(gb_clf, X, y_encoded, cv=5, scoring='accuracy')

# Print mean and standard deviation of accuracy
print(f"Cross-validated accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

Cross-validated accuracy: 0.5371 ± 0.0347


-----

# LdaBoost

In [12]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score


# ---------- utility ----------
def softmax(F):
    expF = np.exp(F - np.max(F, axis=1, keepdims=True))
    return expF / np.sum(expF, axis=1, keepdims=True)


# ---------- model ----------
class CustomMulticlassGradientBoostingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

        # attributes that will hold the learned state (with trailing “_”)
        self.estimators_ = None
        self.lda_transforms_ = None
        self.initial_logit_ = None
        self.classes_ = None

    # -------------------------- training --------------------------
    def fit(self, X, y):
        # reset state
        self.estimators_ = []
        self.lda_transforms_ = []
        self.initial_logit_ = None
        self.classes_ = None

        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]

        prior = np.clip(np.mean(one_hot_y, axis=0), 1e-5, 1 - 1e-5)
        self.initial_logit_ = np.log(prior)

        F = np.tile(self.initial_logit_, (n_samples, 1))

        for m in range(self.n_estimators):
            # 1) LDA
            if m == 0:
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, y)
            else:
                p = softmax(F)
                resid = one_hot_y - p
                labels = np.argmax(resid, axis=1)
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, labels)

            self.lda_transforms_.append(lda)

            # 2) updated residuals
            p = softmax(F)
            resid = one_hot_y - p

            # 3) one tree per class
            estims_m = []
            for k in range(n_classes):
                reg = DecisionTreeRegressor(max_depth=self.max_depth)
                reg.fit(X_lda, resid[:, k])
                estims_m.append(reg)
                F[:, k] += self.learning_rate * reg.predict(X_lda)

            self.estimators_.append(estims_m)

        return self

    # -------------------------- inference --------------------------
    def _decision_function(self, X):
        n_samples = X.shape[0]
        F = np.tile(self.initial_logit_, (n_samples, 1))
        for lda, estims_m in zip(self.lda_transforms_, self.estimators_):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estims_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        return F

    def predict_proba(self, X):
        return softmax(self._decision_function(X))

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ---------- cross-validation ----------
def cross_validate_model(X, y, model, cv=10):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    accs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        model_fold = clone(model)          # now clone works
        model_fold.fit(X_tr, y_tr)
        accs.append(accuracy_score(y_te, model_fold.predict(X_te)))

    return np.mean(accs), np.std(accs)


# ---------- example ----------
model = CustomMulticlassGradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=7
)

X_np = X.to_numpy()


mean_acc, std_acc = cross_validate_model(X_np, y_encoded, model, cv=5)
print(f"Mean accuracy: {mean_acc:.4f} ± {std_acc:.4f}")


Mean accuracy: 0.5728 ± 0.0372
